# Dataset artikel: scraping dan fitur
Dataset utama: Google Top-10 (termasuk yang disitasi Gemini). Dataset tambahan: sitasi Gemini di luar Top-10 untuk query yang sama. Gabungan tetap tersedia sebagai arsip audit. Uji awal mencoba 20 URL unik; URL gagal tetap dicatat. Notebook ini membaca hasil lokal. Menjalankan semua sel dengan pengaturan bawaan tidak mengirim pencarian atau scraping baru.

Ambang contoh disetujui peneliti: proporsi sitasi >=0,5. Status accepted pada artikel merupakan review isi/jenis halaman, bukan bukti kebenaran informasi. Peringkat sumber provisional atau kosong belum dianggap terverifikasi.

In [ ]:
import sys, csv, json, subprocess
from pathlib import Path
from collections import Counter
from html import escape
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/build_article_dataset.py').exists())
FEATURE_CONFIG = ROOT / 'configs/article_features.json'  # ganti satu path ini untuk batch lain
feature_config = json.loads(FEATURE_CONFIG.read_text(encoding='utf-8-sig'))
SCRAPE_CONFIG = ROOT / feature_config['scrape_config']
scrape_config = json.loads(SCRAPE_CONFIG.read_text(encoding='utf-8-sig'))
OUTPUT = ROOT / 'data/processed' / scrape_config['dataset_id']
def read_csv(path):
    with path.open(encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))
def show(rows, columns, limit=30):
    head = ''.join('<th>'+escape(c)+'</th>' for c in columns)
    body = ''.join('<tr>'+''.join('<td>'+escape(str(r.get(c, '')))+'</td>' for c in columns)+'</tr>' for r in rows[:limit])
    display(HTML('<table><thead><tr>'+head+'</tr></thead><tbody>'+body+'</tbody></table>'))
    print(f'Ditampilkan {min(limit, len(rows))} dari {len(rows)} baris.')
def reload_tables():
    return read_csv(OUTPUT / 'articles.csv'), read_csv(OUTPUT / 'dataset_union.csv')
articles, pairs = reload_tables()
print(json.dumps(json.loads((OUTPUT / 'summary.json').read_text(encoding='utf-8')), ensure_ascii=False, indent=2))


## Dataset utama dan tambahan

Pemisahan berlaku per pasangan query–artikel. `dataset.csv` memuat `google_only` dan `google_and_gemini`; `dataset_gemini_only.csv` memuat `gemini_only`. URL yang sama bisa masuk kelompok berbeda untuk query berbeda. `dataset_union.csv` hanya arsip gabungan. Label 1 berarti proporsi sitasi >=0,5, bukan sekadar pernah disitasi; label kosong tetap belum pasti. `model_ready.csv` hanya dataset utama yang lolos pemeriksaan. Pada tambahan, `ready_for_analysis` menunjukkan kelengkapan pemeriksaan; baris tersebut tetap bukan data latih utama.


In [ ]:
primary_pairs = read_csv(OUTPUT / 'dataset.csv')
supplementary_pairs = read_csv(OUTPUT / 'dataset_gemini_only.csv')
print('Pasangan Google Top-10:', len(primary_pairs))
show(primary_pairs, ['query_text', 'hostname', 'candidate_origin', 'citation_proportion', 'citation_label', 'ready_for_model', 'not_ready_reason'])
print('Pasangan sitasi Gemini di luar Top-10:', len(supplementary_pairs))
show(supplementary_pairs, ['query_text', 'hostname', 'citation_proportion', 'citation_label', 'ready_for_analysis', 'not_ready_reason'])


## Hasil pengambilan dan kelayakan
`crawl_status` menunjukkan hasil teknis terbaru pada dataset. `original_crawl_status` mempertahankan status checkpoint awal, termasuk error ekstraksi yang sudah dipulihkan dari HTML lokal. `robots_unavailable` berarti aturan akses belum dapat diperiksa; bukan bukti halaman hilang. `robots_disallowed` berarti crawler tidak diizinkan.


In [ ]:
show(articles, ['article_id', 'hostname', 'crawl_status', 'original_crawl_status', 'article_review_status', 'word_count', 'extraction_method', 'article_url'])
failed = [r for r in articles if r['crawl_status'] != 'success']
print('URL yang belum berhasil diambil:', len(failed))
show(failed, ['article_url', 'crawl_status', 'http_status', 'error_type', 'robots_status'])


## Periksa isi dan fitur
Ubah ARTICLE_ID untuk memeriksa artikel lain. Fitur badan tulisan tidak menghitung menu/footer; `extraction_method` merekam selector atau fallback yang dipakai. BM25 memakai statistik judul + isi artikel eligible dari Google Top-10; artikel Gemini-only tidak menambah statistik korpus; cosine memakai embedding multibahasa lokal.


In [ ]:
ARTICLE_ID = next((r['article_id'] for r in articles if r['article_id'] == 'article_76a558b721efb94a4c4466a8'), articles[0]['article_id'])  # contoh Alodokter jika tersedia; boleh diganti
article = next(r for r in articles if r['article_id'] == ARTICLE_ID)
display(HTML('<h3>'+escape(article['title'])+'</h3><pre style="white-space:pre-wrap">'+escape(article['text'])+'</pre>'))
show([r for r in pairs if r['article_id'] == ARTICLE_ID], ['query_text', 'word_count', 'paragraph_count', 'heading_count', 'list_count', 'wps', 'cpw', 'numeric_mention_count', 'external_reference_count', 'has_author', 'publication_age_days', 'bm25', 'semantic_cosine', 'citation_label', 'ready_for_model', 'not_ready_reason'])


## Kredibilitas dan label
Ranking mengikuti rubrik 1 tertinggi sampai 5 terendah. Kosong berarti bukti/pemetaan rubrik belum cukup, bukan tingkat 5. `provisional` berarti peringkat sementara dan tidak masuk model_ready. Bukti dan alasan ada dalam tabel sumber. Koreksi hasil review pada CSV manual lalu bangun ulang dataset; pengunduhan halaman tidak diperlukan.


In [ ]:
credibility = read_csv(ROOT / feature_config['credibility_csv'])
show(credibility, ['hostname', 'credibility_level', 'review_status', 'reason', 'evidence_urls'])
print('Label gabungan audit:', dict(Counter(r['citation_label'] or 'belum_pasti' for r in pairs)))
ready = [r for r in pairs if r['ready_for_model'] == 'True']
print('Pasangan dataset utama siap model:', len(ready))
show(pairs, ['dataset_group', 'query_text', 'hostname', 'citation_proportion', 'citation_label', 'credibility_level', 'credibility_status', 'ready_for_model', 'not_ready_reason'])


## Melanjutkan sendiri
Untuk mengambil 20 URL baru berikutnya, ubah RUN_SCRAPING menjadi True dan jalankan sel di bawah. MAX_NEW_URLS adalah batas tambahan pada eksekusi ini, bukan total kumulatif. Setelah scraping, builder otomatis memperbarui dataset utama Google Top-10 dan tambahan Gemini-only dari seluruh checkpoint batch ini. Untuk membangun ulang fitur setelah koreksi CSV review, cukup aktifkan RUN_BUILD. Tidak ada penggunaan SerpApi atau Gemini; embedding dihitung lokal dan model diunduh sekali jika belum tersedia.
Untuk batch lain, ganti hanya FEATURE_CONFIG pada sel pemuatan ke config fitur batch tersebut, lalu jalankan ulang sel pemuatan. Config scraping dan folder hasil otomatis mengikuti config fitur itu; pastikan batch sudah mempunyai hasil sebelum membuka tabelnya.

Setelah dijalankan, kembalikan sakelar ke False supaya Run All berikutnya tidak menambah scraping. Untuk mengulang URL gagal secara sengaja, gunakan terminal: python src/scrape_articles.py --run --retry-failed --max-new-urls 3. Perintah retry memilih status gagal sesuai urutan manifest dan tetap memeriksa robots; riwayat tidak ditimpa. Error ekstraksi dengan HTML tersedia cukup dipulihkan lewat RUN_BUILD.


In [ ]:
RUN_SCRAPING = False
RUN_BUILD = False
MAX_NEW_URLS = 20
if RUN_SCRAPING:
    subprocess.run([sys.executable, '-B', str(ROOT / 'src/scrape_articles.py'), '--config', str(SCRAPE_CONFIG), '--run', '--max-new-urls', str(MAX_NEW_URLS)], cwd=ROOT, check=True)
if RUN_BUILD or RUN_SCRAPING:
    subprocess.run([sys.executable, '-B', str(ROOT / 'src/build_article_dataset.py'), '--config', str(FEATURE_CONFIG), '--with-embeddings'], cwd=ROOT, check=True)
    articles, pairs = reload_tables()
    primary_pairs = read_csv(OUTPUT / 'dataset.csv')
    supplementary_pairs = read_csv(OUTPUT / 'dataset_gemini_only.csv')
    print('Dataset utama:', len(primary_pairs), '| dataset tambahan:', len(supplementary_pairs))
    print('Folder hasil:', OUTPUT)
else:
    print('Tidak ada pengambilan atau pembangunan ulang; tabel lokal tetap dapat ditinjau.')
